# 1. Geospatial data: rasters, vectors, formats: exercises

Exercises of [chapter 1](../../01_geospatial_data.md). Run the setup cell first.

Work in a copy of this notebook, under [`docs/manual/solutions/`](..), in a folder `solutions_<you>`: it is regenerated by [`docs/manual/tools/build_notebooks.py`](../../tools/build_notebooks.py), and its links are written for that folder.

In [ ]:
# Setup: run this cell first, in every notebook
import os, subprocess, sys
from pathlib import Path

# find the repository: from this folder upwards, the first one holding data_acquisition/
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "data_acquisition").is_dir())
sys.path.insert(0, str(ROOT))                  # so that Python can import the pipeline
SAMPLE = ROOT / "output/manual/sample_root"    # the course's invented dataset
if not SAMPLE.exists():                        # generated once, with this same Python
    subprocess.run([sys.executable, str(ROOT / "docs/manual/tools/make_sample.py"), "--out", str(SAMPLE)], check=True)
DATASET = SAMPLE / "repo/doi_10.5555_training.pond"    # the sample's dataset folder
EXTRACT = ROOT / "docs/manual/data/montana_extract"    # a real piece of Rumney Ranch
os.environ["BEAVER_SITES"] = str(SAMPLE / "sites.yaml")    # use the sample's site registry, not the real one
import warnings; warnings.simplefilter("ignore")           # hide warnings; remove this line when debugging
print("repository:", ROOT); print("sample data root:", SAMPLE)

In [ ]:
import json
import matplotlib.pyplot as plt
import numpy as np
import pyogrio
import rasterio
from data_acquisition.readers import files, geotiff

**1. A raster's header.** Open the sample's `ortho_2023-07-12.tif` with rasterio and print its width, height, band count, data types, resolution, geotransform, colour interpretation and nodata. Explain each.

> **From the chapter.** §1: a raster is a grid of pixels with one number per band; a CRS and a geotransform place it on the ground.
>
> **Code.** No pipeline code: `rasterio` only. `DATASET` is the sample's dataset folder, from the setup cell. The pipeline's own reader for these values comes in [chapter 13](../../13_readers.md).

In [ ]:
with rasterio.open(DATASET / "ortho_2023-07-12.tif") as r:
    ...  # your code here

**2. Nodata and alpha.** How many pixels of the sample orthomosaic are transparent (alpha 0)? How many pixels of the sample DSM are nodata? Read the DSM with `masked=True` and compare the minimum with and without the mask.

> **From the chapter.** §2: empty pixels are marked either by a nodata value or by an alpha band, so the two files need different code.
>
> **Code.** No pipeline code: `rasterio` and NumPy. In the pipeline, `curate.check_raster` in [`data_acquisition/steps/curate.py`](../../../../data_acquisition/steps/curate.py) flags a raster that has neither ([chapter 15](../../15_curate.md)).

In [ ]:
...  # your code here

**3. Vectors.** List the layers of the sample's `dams_2023-07-12.gpkg` and of the extract's `2018-07-26/annotations/dams.gpkg`: geometry type, number of features, attribute names. Compare the geometry type pyogrio reports with the one `ogrinfo` reports for the extract's layer: what is lost?

> **From the chapter.** §3: a file holds layers, a layer holds features, and coordinates can carry Z and M. Not every tool keeps M.
>
> **Code.** No pipeline code: the library `pyogrio`, and the GDAL command-line tool `ogrinfo` ([chapter 0](../../00_setup.md), §3), run with `subprocess`. `EXTRACT` is the Montana extract, from the setup cell.

In [ ]:
...  # your code here (pyogrio.list_layers, pyogrio.read_info, subprocess ogrinfo -json -so)

**4. Companions.** Section 5 of the chapter showed that a raster or a shapefile can come with small files sharing its name. The pipeline decides which files belong together with `files.companions(name)`, from [`data_acquisition/readers/files.py`](../../../../data_acquisition/readers/files.py) (the notebook imports it as `files`). It works on the name only and never looks at the disk, so the file does not need to exist. Call it on `"x.tif"` and on `"dams.shp"`, then answer:

- Which of the names returned for `x.tif` are in the table of section 5, and which are not?
- Why does the same sidecar appear in two forms, such as `x.aux.xml` and `x.tif.aux.xml`?
- What could go wrong if you copied `x.tif` to another folder without `x.tif.aux.xml`?

In [ ]:
...  # your code here (files.companions)

**5. Draw both.** Plot the extract's 26 July orthomosaic chunk and draw the dam outlines on it.

> **From the chapter.** §1 and §3, and the Case: the raster's geotransform places its pixels, and the dams are already in map coordinates. `r.transform` goes from pixel to map, and `~r.transform` from map to pixel; both work only in the raster's CRS, so convert the dams first. The dams were drawn on the chunk, so use `orthomosaic_chunk.tif`: [chapter 3](../../03_drone_surveys.md) explains why they do not fit the full tile.
>
> **Code.** No pipeline code: `rasterio`, `pyogrio` and `matplotlib`, on `EXTRACT / "2018-07-26/orthomosaic_chunk.tif"` and `EXTRACT / "2018-07-26/annotations/dams.gpkg"`.

In [ ]:
...  # your code here (rasterio, pyogrio.read_dataframe, matplotlib)

**6. Find it in the code.** Which function decides whether a file is a raster, and which file extensions does it try before opening a file?

> **From the chapter.** §4 lists the raster formats; [chapter 13](../../13_readers.md), §1 explains how the pipeline recognises them.
>
> **Code.** Look in [`data_acquisition/readers/geotiff.py`](../../../../data_acquisition/readers/geotiff.py) (imported above as `geotiff`). `inspect.getsource` prints a function or a module; a constant is a plain value, so print it directly.

In [ ]:
import inspect
...  # your code here